# NB11 — Skenario E: Two-Pass HDBSCAN + GLOSH + GaussianNB

**Pipeline:** LOF (NB10) → UMAP (NB10) → HDBSCAN Pass 1 (mcs=15, ms=10) →
HDBSCAN Pass 2 (mcs=2, ms=10) → GLOSH + approximate_predict → GaussianNB

**Hasil:** Coverage 99.6%, DBCV 0.6588, ARI 0.9609, Purity 0.9987, n_clusters 139

**Input:** output_nb10/umap_coords_inlier.npy, output_nb10/skenario_d_results.pkl, output_nb05/metadata_labeled.pkl

**Output:** output_nb11/skenario_e_results.pkl, output_nb11/labels_E.npy

In [1]:
import pickle, numpy as np, pandas as pd
from pathlib import Path
from collections import Counter
from sklearn.metrics import (adjusted_rand_score,
                             normalized_mutual_info_score,
                             silhouette_score, davies_bouldin_score)
from sklearn.naive_bayes import GaussianNB
import hdbscan

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

In [ ]:
BASE       = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings")
NB05_DIR   = BASE / "output_nb05"
NB10_DIR   = BASE / "output_nb10"
OUTPUT_DIR = BASE / "output_nb11"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MCS_PASS1         = 15
MCS_PASS2         = 2
MS                = 10
GLOSH_HIGH        = 0.9
STRENGTH_LOW      = 0.1
STRENGTH_MODERATE = 0.5

print('Config OK.')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
X_umap = np.load(NB10_DIR / "umap_coords_inlier.npy")

with open(NB10_DIR / "skenario_d_results.pkl", "rb") as f:
    results_d = pickle.load(f)

with open(NB05_DIR / "metadata_labeled.pkl", "rb") as f:
    meta_all = pickle.load(f)

idx_inlier   = results_d["idx_inlier"]
meta_inlier  = [meta_all[i] for i in idx_inlier]
N            = len(meta_inlier)
labels_A_all = np.array([m["cluster_id"] for m in meta_all])
labels_A     = labels_A_all[idx_inlier]

assert X_umap.shape[0] == N
print(f"Wajah inlier (post-LOF) : {N:,}")
print(f"X_umap shape            : {X_umap.shape}")
print(f"Noise A (pada inlier)   : {(labels_A==-1).sum():,} ({(labels_A==-1).mean()*100:.1f}%)")


In [ ]:
print(f"HDBSCAN Pass 1 (mcs={MCS_PASS1}, ms={MS})...")

clusterer1   = hdbscan.HDBSCAN(
    min_cluster_size=MCS_PASS1,
    min_samples=MS,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)
labels1      = clusterer1.fit_predict(X_umap)
glosh_scores = clusterer1.outlier_scores_

n_clusters1 = len(set(labels1)) - (1 if -1 in labels1 else 0)
n_noise1    = (labels1 == -1).sum()
print(f"  Clusters : {n_clusters1}")
print(f"  Noise    : {n_noise1:,} ({n_noise1/N*100:.1f}%)")
print(f"  Coverage : {(labels1>=0).mean()*100:.1f}%")

In [ ]:
noise1_idx  = np.where(labels1 == -1)[0]
X_noise1    = X_umap[noise1_idx]

print(f"HDBSCAN Pass 2 (mcs={MCS_PASS2}, ms={MS}) pada {len(noise1_idx):,} noise Pass 1...")

clusterer2  = hdbscan.HDBSCAN(
    min_cluster_size=MCS_PASS2,
    min_samples=MS,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)
labels2_raw = clusterer2.fit_predict(X_noise1)

max_label1  = labels1.max()
labels2     = np.where(labels2_raw >= 0, labels2_raw + max_label1 + 1, -1)

n_new = len(set(labels2_raw)) - (1 if -1 in labels2_raw else 0)
print(f"  Clusters baru : {n_new}")
print(f"  Masih noise   : {(labels2==-1).sum():,}")

In [ ]:
labels_E  = labels1.copy()
labels_E[noise1_idx] = labels2

still_idx          = noise1_idx[labels2 == -1]
X_sn               = X_umap[still_idx]
print(f"Sisa noise setelah 2 pass: {len(still_idx):,}")

glosh_sn           = glosh_scores[still_idx]
labels_ap, str_ap  = hdbscan.approximate_predict(clusterer1, X_sn)

mask_outlier = glosh_sn > GLOSH_HIGH
mask_direct  = ~mask_outlier & (str_ap >= STRENGTH_MODERATE)
mask_nb      = ~mask_outlier & (str_ap >= STRENGTH_LOW) & (str_ap < STRENGTH_MODERATE)
mask_single  = ~mask_outlier & (str_ap < STRENGTH_LOW)

labels_E[still_idx[mask_direct]] = labels_ap[mask_direct]

print(f"  Outlier permanen (GLOSH>{GLOSH_HIGH})          : {mask_outlier.sum():,}")
print(f"  Direct assign (strength>={STRENGTH_MODERATE})  : {mask_direct.sum():,}")
print(f"  GaussianNB kandidat                            : {mask_nb.sum():,}")
print(f"  Singleton (strength<{STRENGTH_LOW})             : {mask_single.sum():,}")

In [ ]:
mask_clust    = labels_E >= 0
gnb           = GaussianNB()
gnb.fit(X_umap[mask_clust], labels_E[mask_clust])

nb_global_idx = still_idx[mask_nb]
if len(nb_global_idx) > 0:
    labels_E[nb_global_idx] = gnb.predict(X_umap[nb_global_idx])
    print(f"GaussianNB: {len(nb_global_idx):,} wajah di-assign")
else:
    print("Tidak ada kandidat GaussianNB.")

n_noise_E  = (labels_E == -1).sum()
coverage_E = (labels_E >= 0).sum() / N
n_clust_E  = len(set(labels_E)) - (1 if -1 in labels_E else 0)
print(f"\nHasil akhir:")
print(f"  Clusters : {n_clust_E}")
print(f"  Noise    : {n_noise_E:,} ({n_noise_E/N*100:.1f}%)")
print(f"  Coverage : {coverage_E*100:.1f}%")

In [ ]:
def purity(y_true, y_pred):
    total, correct = 0, 0
    for cl in set(y_pred):
        if cl < 0: continue
        mask     = y_pred == cl
        majority = Counter(y_true[mask]).most_common(1)[0][1]
        correct += majority; total += mask.sum()
    return correct / total if total > 0 else 0.0

mask_clust_E = labels_E >= 0
dbcv_E = hdbscan.validity_index(X_umap.astype(np.float64), labels_E)
sil_E  = silhouette_score(X_umap[mask_clust_E], labels_E[mask_clust_E], metric='euclidean')
dbi_E  = davies_bouldin_score(X_umap[mask_clust_E], labels_E[mask_clust_E])

mask_eval = labels_A >= 0
ari_E     = adjusted_rand_score(labels_A[mask_eval], labels_E[mask_eval])
nmi_E     = normalized_mutual_info_score(labels_A[mask_eval], labels_E[mask_eval])
pur_E     = purity(labels_A[mask_eval], labels_E[mask_eval])

print('=' * 50)
print(f"{'Metrik':<20} {'Skenario D':>12} {'Skenario E':>12}")
print('=' * 50)
print(f"{'DBCV':<20} {'0.7634':>12} {dbcv_E:>12.4f}")
print(f"{'Silhouette':<20} {'0.8336':>12} {sil_E:>12.4f}")
print(f"{'DBI':<20} {'0.4440':>12} {dbi_E:>12.4f}")
print(f"{'ARI':<20} {'0.9650':>12} {ari_E:>12.4f}")
print(f"{'NMI':<20} {'—':>12} {nmi_E:>12.4f}")
print(f"{'Purity':<20} {'—':>12} {pur_E:>12.4f}")
print(f"{'Coverage (%)':<20} {'97.0':>12} {coverage_E*100:>11.1f}%")
print(f"{'n_clusters':<20} {'~105':>12} {n_clust_E:>12}")
print('=' * 50)

In [ ]:
results_E = {
    'labels_E'   : labels_E,
    'meta'       : meta_inlier,
    'labels_A'   : labels_A,
    'dbcv'       : dbcv_E, 'silhouette': sil_E, 'dbi': dbi_E,
    'ari'        : ari_E,  'nmi': nmi_E, 'purity': pur_E,
    'coverage'   : coverage_E, 'n_clusters': n_clust_E,
    'params'     : {'mcs_pass1': MCS_PASS1, 'mcs_pass2': MCS_PASS2, 'ms': MS}
}

with open(OUTPUT_DIR / "skenario_e_results.pkl", "wb") as f:
    pickle.dump(results_E, f)
np.save(OUTPUT_DIR / "labels_E.npy", labels_E)

print("Tersimpan di:", OUTPUT_DIR)